In [55]:
from pathlib import Path

import sqlite3
import pandas as pd


path = Path("data")

In [56]:
csv_files = list(path.glob("*.csv"))
dfs = [pd.read_csv(file) for file in csv_files]

In [57]:
dfs[5].head()

,category_id,name,description
0,1,Guitarras,"Guitarras elétricas de todos os estilos, do ro..."
1,2,Baixos,Baixos elétricos e acústicos para todas as for...
2,3,Baterias e Percussão,"Baterias acústicas, eletrônicas e peças avulsa..."
3,4,Teclados e Pianos,"Pianos digitais, sintetizadores, arranjadores ..."
4,5,Violões,"Violões clássicos, folk e eletroacústicos"


In [58]:
conn = sqlite3.connect("data/database.db")

for f in csv_files:
    df = pd.read_csv(f)

    table_name = f.stem.split(" - ", maxsplit=1)[1]  # Nome do arquivo sem extensão
    
    df.to_sql(
        name=table_name,
        con=conn,
        if_exists="replace",
        index=False
    )

conn.close()

In [59]:
def query_db(query: str) -> pd.DataFrame:
    with sqlite3.connect("data/database.db") as conn:
        return pd.read_sql_query(query, conn)

In [60]:
query = """
SELECT
    o.order_id,
    c.name AS customer_name,
    c.city,
    o.order_date,
    o.status,
    o.total_brl,
    o.payment_method
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
ORDER BY o.order_date DESC;
"""

df = query_db(query)

In [61]:
df

,order_id,customer_name,city,order_date,status,total_brl,payment_method
0,16,Camila Duarte Nascimento,Campo Grande,2026-03-22,pending,429.9,credit_6x
1,15,Aline Borges Monteiro,Campo Grande,2026-03-18,pending,8348.0,credit_12x
2,14,Diego Fernandes Castro,Campo Grande,2026-03-15,pending,1749.0,debit
3,13,Fernanda Ribeiro Souza,Campo Grande,2026-03-12,pending,1899.0,pix
4,12,Patrícia Vieira Cardoso,Campo Grande,2026-03-10,confirmed,599.0,credit_3x
5,20,Thiago Barbosa Lima,Campo Grande,2026-03-08,cancelled,1400.0,credit_3x
6,11,Mariana Costa Santos,Campo Grande,2026-03-05,confirmed,359.0,boleto
7,10,Bruno Carvalho Martins,Três Lagoas,2026-02-25,confirmed,8732.0,pix
8,19,Pedro Henrique Oliveira,Campo Grande,2026-02-22,cancelled,7499.0,boleto
9,9,Juliana Almeida Rodrigues,Campo Grande,2026-02-20,confirmed,349.9,credit_6x


In [62]:
query = """
SELECT
    p.product_id,
    p.name AS product_name,
    c.name AS category,
    SUM(oi.quantity) AS units_sold,
    SUM(oi.quantity * p.price_brl) AS estimated_revenue
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
JOIN categories c
    ON p.category_id = c.category_id
JOIN orders o
    ON oi.order_id = o.order_id
WHERE o.status != 'cancelled'
GROUP BY p.product_id, p.name, c.name
ORDER BY units_sold DESC;
"""

df = query_db(query)

In [63]:
df

,product_id,product_name,category,units_sold,estimated_revenue
0,130,Kala KA-ABP-CTG Barítono Cedar,Ukuleles,3,5247.0
1,106,Taylor 214ce Elétrico Aço Natural,Violões,2,14998.0
2,120,Kala KA-C Concert Mogno,Ukuleles,2,699.8
3,82,Yamaha C70 Nylon Natural,Violões,1,849.0
4,83,Tagima Memphis AC-39 Nylon Natural,Violões,1,429.9
5,93,Martin D-28 Dreadnought Natural,Violões,1,11499.0
6,94,Taylor 110e Dreadnought Natural,Violões,1,5999.0
7,96,Giannini GF-3D Dreadnought Sunburst,Violões,1,799.9
8,97,Crafter D-8 Dreadnought Natural,Violões,1,4599.0
9,119,Shelby SU-21S Soprano Sunburst,Ukuleles,1,159.9


In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model="qwen/qwen3.8-27b", # Alternativa leve e mais rápida: "llama3-8b-8192"
    temperature=0.2
)

resposta = llm.invoke("Você vende guitarras?")
print(resposta.content)

Não, eu sou uma inteligência artificial, então não vendo produtos físicos como guitarras.

No entanto, posso te ajudar de outras formas:
*   **Recomendar modelos** de guitarra com base no seu estilo musical (rock, jazz, blues, etc.), orçamento e nível de experiência.
*   **Explicar diferenças** entre tipos de madeira, captadores (single coil vs. humbucker) e configurações.
*   **Dar dicas** de manutenção, afinação ou técnicas de dedilhado.
*   **Sugerir onde procurar** boas ofertas ou lojas confiáveis na sua região.

Se quiser, me conte um pouco mais sobre o que você procura! 🎸
